# 🧬 Cancer Prediction Using Machine Learning

## Phase 2 — Dataset Understanding

**Objective:** Load the Wisconsin Breast Cancer dataset, understand every feature,
identify data quality issues, and prepare for cleaning.

---

### 📋 Table of Contents

1. Setup & Imports
2. Load the Dataset
3. Dataset Overview
4. Column Descriptions
5. Data Types Analysis
6. Missing Values
7. Duplicate Values
8. Target Variable Analysis
9. Basic Statistics
10. Outlier Detection (Initial)
11. Phase 2 Summary

---

## 1. 🔧 Setup & Imports

### Why do we import modules at the top?

PEP-8 (Python's style guide) mandates that all imports go at the top of the file.
This makes it immediately clear what dependencies a script has, and if something
is missing, you'll know right away — not 20 minutes into execution.

**Import order (PEP-8 convention):**
1. Standard library (`os`, `sys`, `logging`)
2. Third-party packages (`pandas`, `numpy`, `matplotlib`)
3. Local modules (`src/data_loader.py`, `src/utils.py`)

In [ ]:
# =============================================================================
# CELL 1: Setup & Imports
# =============================================================================
# WHY: Import all required libraries upfront so any missing package
#       is caught immediately, not mid-analysis.
#
# NOTE: We add the src/ directory to Python's path so we can import
#       our custom modules (data_loader, utils) cleanly.
# =============================================================================

# --- Standard Library ---
import os
import sys
import warnings

# Suppress warnings for cleaner notebook output
warnings.filterwarnings('ignore')

# --- Third-Party Libraries ---
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- Configure Display Settings ---
# Show all columns when printing DataFrames (don't truncate)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)
pd.set_option('display.float_format', '{:.4f}'.format)

# Matplotlib inline rendering
%matplotlib inline
plt.style.use('seaborn-v0_8-darkgrid')  # Professional chart style
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12

# --- Add src/ to import path ---
# This lets us import our custom modules from the src/ folder
src_path = os.path.abspath(os.path.join('..', 'src'))
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from data_loader import load_from_sklearn, save_raw_data, get_feature_descriptions
from utils import setup_logging, print_section_header

# Setup logging
logger = setup_logging()

print("✅ All imports successful!")
print(f"📁 Working directory: {os.getcwd()}")
print(f"🐍 Python version: {sys.version.split()[0]}")
print(f"📊 Pandas version: {pd.__version__}")
print(f"🔢 NumPy version: {np.__version__}")

### 💡 Best Practice: `warnings.filterwarnings('ignore')`

We suppress warnings in notebooks for cleaner output. In **production code**, you should
**never** suppress warnings globally — instead, handle them explicitly.

**🎯 Interview Question:** *What is the correct PEP-8 import order?*
> 1. Standard library imports (`os`, `sys`)
> 2. Third-party imports (`pandas`, `numpy`)
> 3. Local/project imports (`from data_loader import ...`)
> Each group is separated by a blank line.

---

## 2. 📦 Load the Dataset

### About the Wisconsin Breast Cancer Diagnostic Dataset

| Property | Value |
|----------|-------|
| **Source** | UCI Machine Learning Repository |
| **Creator** | Dr. William H. Wolberg, University of Wisconsin |
| **Samples** | 569 |
| **Features** | 30 numeric (real-valued) |
| **Target** | Diagnosis: M (Malignant) or B (Benign) |
| **Missing Values** | None |
| **Year** | 1995 |

### How the data was collected

A **Fine Needle Aspirate (FNA)** biopsy was performed on breast masses. The cells were
stained and digitized. A computer program called **Xcyt** computed 10 features for each
cell nucleus in the image. For each of the 10 features, three statistics were computed:

- **Mean** — average value across all nuclei in the image
- **SE** (Standard Error) — how much the measurements vary
- **Worst** — mean of the 3 largest (most extreme) nuclei

This gives us 10 × 3 = **30 features** total.

### Why sklearn's version?

We use `sklearn.datasets.load_breast_cancer()` because:
1. **Reproducible** — same dataset on every machine
2. **No download required** — bundled with sklearn
3. **Well-documented** — feature names and descriptions included
4. **Clean** — no missing values or formatting issues

In [ ]:
# =============================================================================
# CELL 2: Load the Dataset
# =============================================================================
# WHY: We load from sklearn for reproducibility, then immediately save
#       a raw CSV copy. The raw copy is our "golden source" — we NEVER
#       modify it. All transformations go to data/processed/.
#
# BEST PRACTICE: Always keep raw data separate from processed data.
#       If your cleaning pipeline has a bug, you can always start over
#       from the raw data without re-downloading.
# =============================================================================

print_section_header("Loading Wisconsin Breast Cancer Dataset", "📦")

# Load from sklearn using our data_loader module
df = load_from_sklearn()

# Save raw data — this is our untouched golden copy
raw_path = save_raw_data(df)

print(f"\n📊 Dataset loaded successfully!")
print(f"   Rows:    {df.shape[0]}")
print(f"   Columns: {df.shape[1]}")
print(f"   💾 Raw data saved to: {raw_path}")
print(f"\n🔍 First 5 rows:")
df.head()

### 💡 Why save raw data immediately?

In industry, you **always** preserve the original data:

1. **Audit trail** — regulators may ask "what was the original data?"
2. **Bug recovery** — if your preprocessing has a bug, you can restart
3. **Reproducibility** — teammates can verify your starting point
4. **Data versioning** — compare raw vs. processed to document changes

**Common beginner mistake:** Modifying data in-place without keeping the original. 
This makes it impossible to debug preprocessing errors.

**🎯 Interview Question:** *What is the difference between raw data and processed data in an ML pipeline?*
> Raw data is the original, unmodified dataset as received from the source. Processed data has been cleaned, transformed, encoded, and prepared for model training. You should always keep both versions for reproducibility.

---

## 3. 🔬 Dataset Overview

Before any analysis, we need to understand the **shape, structure, and types** of our data.
This is like a doctor examining a patient before diagnosing — you need to understand
what you're working with.

We'll use three key pandas methods:
- `.shape` — how many rows and columns?
- `.info()` — column names, types, and non-null counts
- `.describe()` — statistical summary of numeric columns

In [ ]:
# =============================================================================
# CELL 3: Dataset Shape & Info
# =============================================================================
# WHY: .info() gives us a complete picture of the DataFrame:
#       - Column names and their order
#       - Data type of each column (float64, int64, object)
#       - Non-null count (helps spot missing values)
#       - Memory usage
#
# TIME COMPLEXITY: O(n) where n = number of columns — negligible
# =============================================================================

print_section_header("Dataset Shape", "📐")
print(f"  Rows (samples):    {df.shape[0]}")
print(f"  Columns (features): {df.shape[1]}")
print(f"  Total data points:  {df.shape[0] * df.shape[1]:,}")
print(f"  Memory usage:       {df.memory_usage(deep=True).sum() / 1024:.1f} KB")

print_section_header("DataFrame Info", "ℹ️")
df.info()

In [ ]:
# =============================================================================
# CELL 4: Statistical Summary
# =============================================================================
# WHY: .describe() shows count, mean, std, min, 25%, 50%, 75%, max
#       for every numeric column. This helps us:
#       1. Spot features with very different scales (need normalization)
#       2. Identify potential outliers (large gap between 75% and max)
#       3. Check if any feature has zero variance (useless for ML)
#
# OBSERVATION: Notice how 'area_mean' ranges from 143 to 2501, while
#       'smoothness_mean' ranges from 0.05 to 0.16. These features are
#       on completely different scales — we'll need StandardScaler later.
# =============================================================================

print_section_header("Statistical Summary", "📊")
df.describe().T  # Transpose for better readability with many columns

### 💡 Why transpose `.describe().T`?

With 30+ columns, `.describe()` creates a very wide table. Transposing it (`.T`) puts
features as rows and statistics as columns, making it much easier to scan vertically.

### Key observations from `.describe()`:

Look for these patterns:
- **Scale differences**: `area_mean` (143-2501) vs `smoothness_mean` (0.05-0.16) → Need scaling
- **Potential outliers**: Large gap between 75th percentile and max
- **Zero variance**: If std ≈ 0, the feature is useless
- **Negative values**: None expected in this dataset (all measurements are positive)

---

## 4. 📋 Column Descriptions

Understanding what each feature **means** is critical. Many beginners skip this step
and treat features as abstract numbers — this leads to poor feature engineering decisions.

### The 10 base cell nucleus measurements:

| # | Feature | Description | Unit |
|---|---------|-------------|------|
| 1 | **radius** | Mean distance from center to perimeter | μm |
| 2 | **texture** | Standard deviation of gray-scale values | — |
| 3 | **perimeter** | Total length of the cell boundary | μm |
| 4 | **area** | Area enclosed by the cell boundary | μm² |
| 5 | **smoothness** | Local variation in radius lengths | — |
| 6 | **compactness** | perimeter² / area - 1.0 | — |
| 7 | **concavity** | Severity of concave portions | — |
| 8 | **concave points** | Number of concave portions | count |
| 9 | **symmetry** | How symmetric the nucleus is | — |
| 10 | **fractal dimension** | "Coastline approximation" - 1 (complexity) | — |

Each of these 10 features has **3 variants**: `_mean`, `_se` (standard error), `_worst` (mean of 3 largest values).

**Total: 10 × 3 = 30 features + 1 ID + 1 target = 32 columns**

In [ ]:
# =============================================================================
# CELL 5: Display All Columns With Types
# =============================================================================
# WHY: A clear, formatted view of every column helps us identify:
#       - Which columns are features vs metadata (id)
#       - Which is the target (diagnosis)
#       - Data types (all numeric features should be float64)
# =============================================================================

print_section_header("Column Details", "📋")

# Create a summary DataFrame
col_summary = pd.DataFrame({
    'Column': df.columns,
    'Type': df.dtypes.values,
    'Non-Null': df.notnull().sum().values,
    'Null': df.isnull().sum().values,
    'Unique': df.nunique().values,
    'Sample Value': [df[col].iloc[0] for col in df.columns]
})

# Add a role column
def get_role(col_name):
    """Classify column role in the dataset."""
    if col_name == 'id':
        return '🏷️ Identifier'
    elif col_name == 'diagnosis':
        return '🎯 Target'
    else:
        return '📊 Feature'

col_summary['Role'] = col_summary['Column'].apply(get_role)

print(col_summary.to_string(index=False))
print(f"\n📊 Total columns: {len(df.columns)}")
print(f"   Features: {len(df.columns) - 2} (excluding id and diagnosis)")
print(f"   Target: diagnosis")
print(f"   Identifier: id (to be dropped before training)")

### 💡 Understanding feature types

In ML, we categorize features by their data type:

| Type | Description | Example | Encoding needed? |
|------|-------------|---------|------------------|
| **Numeric (Continuous)** | Real-valued numbers | radius=17.99 | No (maybe scaling) |
| **Numeric (Discrete)** | Integer counts | concave_points=3 | No |
| **Categorical (Nominal)** | Unordered categories | color=red/blue | Yes (One-Hot) |
| **Categorical (Ordinal)** | Ordered categories | size=S/M/L | Yes (Label/Ordinal) |
| **Binary** | Two values | diagnosis=M/B | Yes (0/1) |

Our dataset has:
- **30 numeric continuous features** (float64) ← No encoding needed, but need scaling
- **1 binary target** (object/string) ← Needs Label Encoding (M→1, B→0)
- **1 identifier** (int64) ← Drop before training

**🎯 Interview Question:** *What is the difference between nominal and ordinal categorical data?*
> Nominal data has no natural order (colors: red, blue, green). Ordinal data has a meaningful order (sizes: small, medium, large). This distinction matters because ordinal data can use Label Encoding, while nominal data should use One-Hot Encoding to avoid implying a false ordering.

---

## 5. 📊 Data Types Analysis

Verifying data types is crucial because:
- A numeric column stored as `object` (string) won't work with ML algorithms
- An integer column that should be float may lose precision
- Categorical columns need explicit encoding

In [ ]:
# =============================================================================
# CELL 6: Data Types Visualization
# =============================================================================
# WHY: Visual confirmation that data types match our expectations.
#       All features should be float64, target should be object (string),
#       and id should be int64.
# =============================================================================

print_section_header("Data Types Summary", "🔍")

# Count data types
dtype_counts = df.dtypes.value_counts()
print("Data type distribution:")
for dtype, count in dtype_counts.items():
    print(f"  {str(dtype):10s} → {count} columns")

# Verify expectations
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f"\n✅ Numeric columns: {len(numeric_cols)}")
print(f"✅ Categorical columns: {len(categorical_cols)} → {categorical_cols}")

# Check if any numeric column has unexpected values
print("\n🔍 Checking for negative values in features (should be none):")
feature_cols = [col for col in numeric_cols if col != 'id']
negative_check = (df[feature_cols] < 0).sum()
negative_features = negative_check[negative_check > 0]
if len(negative_features) == 0:
    print("  ✅ No negative values found — all measurements are positive as expected")
else:
    print(f"  ⚠️ Negative values found in: {negative_features.to_dict()}")

---

## 6. ❓ Missing Values Analysis

### Why check for missing values?

Missing values are one of the most common data quality issues. They can:
1. **Crash ML algorithms** — most sklearn models can't handle NaN
2. **Bias results** — if data is not missing at random (MNAR)
3. **Reduce sample size** — if you simply drop rows with missing values

### Types of missing data:

| Type | Meaning | Example | Strategy |
|------|---------|---------|----------|
| **MCAR** | Missing Completely At Random | Lab machine error | Safe to drop or impute |
| **MAR** | Missing At Random (depends on other variables) | Income missing for unemployed | Impute with care |
| **MNAR** | Missing Not At Random (depends on the value itself) | High-income people skip income question | Requires domain knowledge |

In [ ]:
# =============================================================================
# CELL 7: Missing Values Check
# =============================================================================
# WHY: We MUST check for missing values before ANY analysis.
#       Even if the dataset documentation says "no missing values",
#       always verify — documentation can be wrong.
#
# BEST PRACTICE: Check both NaN and empty strings. Some datasets
#       use empty strings or special values (like -999) instead of NaN.
# =============================================================================

print_section_header("Missing Values Analysis", "❓")

# Check for NaN values
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_summary = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct
}).sort_values('Missing Count', ascending=False)

# Show only columns with missing values (if any)
cols_with_missing = missing_summary[missing_summary['Missing Count'] > 0]

if len(cols_with_missing) == 0:
    print("✅ No missing values found in any column!")
    print(f"   All {df.shape[0]} rows are complete across all {df.shape[1]} columns.")
else:
    print("⚠️ Columns with missing values:")
    print(cols_with_missing)

# Also check for empty strings (hidden missing values)
print("\n🔍 Checking for empty strings:")
empty_strings = (df == '').sum()
if empty_strings.sum() == 0:
    print("  ✅ No empty strings found")
else:
    print(f"  ⚠️ Empty strings found: {empty_strings[empty_strings > 0].to_dict()}")

# Check for infinite values in numeric columns
print("\n🔍 Checking for infinite values:")
inf_check = np.isinf(df.select_dtypes(include=[np.number])).sum()
if inf_check.sum() == 0:
    print("  ✅ No infinite values found")
else:
    print(f"  ⚠️ Infinite values found: {inf_check[inf_check > 0].to_dict()}")

print(f"\n📊 Data completeness: {(1 - df.isnull().sum().sum() / df.size) * 100:.2f}%")

### 💡 Analysis

The Wisconsin Breast Cancer dataset is very clean — **no missing values**. This is rare
in real-world data. Most production datasets have 5-30% missing values.

Even though we have no missing values here, the **process** of checking is critical.
In your next project, you'll likely need to handle missing data using strategies like:

- **Drop rows** — when < 5% of data is missing and it's MCAR
- **Mean/Median imputation** — for numeric features
- **Mode imputation** — for categorical features
- **KNN imputation** — uses similar rows to fill gaps
- **Create indicator variable** — add a `feature_was_missing` column

**🎯 Interview Question:** *How do you handle missing values in a dataset?*
> First, I analyze the pattern of missingness (MCAR/MAR/MNAR). For small amounts of MCAR data, dropping rows is acceptable. For larger amounts, I use imputation — median for numeric features (robust to outliers) and mode for categorical. I also consider creating a binary indicator column to preserve the information that a value was missing.

---

## 7. 🔄 Duplicate Values

### Why check for duplicates?

Duplicate rows can:
1. **Inflate model performance** — if the same sample appears in train and test
2. **Bias the model** — over-representing certain patterns
3. **Indicate data collection issues** — double-recorded patients

In [ ]:
# =============================================================================
# CELL 8: Duplicate Values Check
# =============================================================================
# WHY: Duplicates can leak information and bias model training.
#       We check for:
#       1. Exact duplicates (all columns match)
#       2. Feature duplicates (same features but different ID)
#       3. ID duplicates (same patient recorded twice)
# =============================================================================

print_section_header("Duplicate Values Analysis", "🔄")

# Check 1: Exact duplicates (all columns)
exact_dupes = df.duplicated().sum()
print(f"  Exact duplicate rows: {exact_dupes}")

# Check 2: Feature duplicates (excluding ID column)
feature_dupes = df.drop(columns=['id']).duplicated().sum()
print(f"  Feature duplicates (excl. ID): {feature_dupes}")

# Check 3: Duplicate IDs
id_dupes = df['id'].duplicated().sum()
print(f"  Duplicate IDs: {id_dupes}")

if exact_dupes == 0 and feature_dupes == 0 and id_dupes == 0:
    print("\n✅ No duplicates found — every row is unique!")
else:
    print("\n⚠️ Duplicates detected — will need to handle in Phase 3 (Data Cleaning)")
    if feature_dupes > 0:
        print(f"   {feature_dupes} rows have identical features but different IDs")
        print("   Showing duplicate feature rows:")
        dupe_mask = df.drop(columns=['id']).duplicated(keep=False)
        print(df[dupe_mask].head(10))

---

## 8. 🎯 Target Variable Analysis

### Why is this critical?

Understanding the **class distribution** of our target variable determines:
- Whether we have a **class imbalance** problem
- Which **metrics** are appropriate (accuracy is misleading for imbalanced data)
- Whether we need **resampling** techniques (SMOTE, undersampling)

### What is class imbalance?

If one class has significantly more samples than the other (e.g., 95% Benign, 5% Malignant),
a model can achieve 95% accuracy by simply predicting "Benign" every time — without
actually learning anything. This is why accuracy alone is a dangerous metric.

In [ ]:
# =============================================================================
# CELL 9: Target Variable Analysis
# =============================================================================
# WHY: We must understand the class distribution before training.
#       Imbalanced classes require special handling (SMOTE, class weights,
#       stratified splitting, appropriate metrics).
#
# BUSINESS CONTEXT: In cancer diagnosis, we expect more Benign cases
#       than Malignant ones, which is consistent with real-world data.
# =============================================================================

print_section_header("Target Variable: diagnosis", "🎯")

# Value counts
target_counts = df['diagnosis'].value_counts()
target_pct = df['diagnosis'].value_counts(normalize=True) * 100

print("Class Distribution:")
print(f"  B (Benign):     {target_counts.get('B', 0)} samples ({target_pct.get('B', 0):.1f}%)")
print(f"  M (Malignant):  {target_counts.get('M', 0)} samples ({target_pct.get('M', 0):.1f}%)")
print(f"  Ratio (B:M):    {target_counts.get('B', 0) / target_counts.get('M', 1):.2f} : 1")

# Assess imbalance
ratio = target_counts.max() / target_counts.min()
if ratio > 3:
    print(f"\n⚠️ SIGNIFICANT class imbalance detected (ratio: {ratio:.1f}:1)")
    print("   Consider: SMOTE, class weights, or stratified sampling")
elif ratio > 1.5:
    print(f"\n⚠️ MODERATE class imbalance detected (ratio: {ratio:.1f}:1)")
    print("   Will use stratified train-test split to preserve proportions")
else:
    print(f"\n✅ Classes are reasonably balanced (ratio: {ratio:.1f}:1)")

In [ ]:
# =============================================================================
# CELL 10: Target Distribution Visualization
# =============================================================================
# WHY: Visual representation makes class imbalance immediately obvious.
#       We use two charts: a bar chart (exact counts) and a pie chart
#       (proportions at a glance).
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Color palette: green for Benign (safe), red for Malignant (danger)
colors = {'B': '#3fb950', 'M': '#f85149'}
color_list = [colors[x] for x in target_counts.index]

# --- Bar Chart ---
bars = axes[0].bar(target_counts.index, target_counts.values, 
                   color=color_list, edgecolor='white', linewidth=2, width=0.5)

# Add value labels on bars
for bar, count, pct in zip(bars, target_counts.values, target_pct.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
                f'{count}\n({pct:.1f}%)', ha='center', va='bottom',
                fontsize=13, fontweight='bold')

axes[0].set_title('Diagnosis Distribution (Count)', fontsize=14, fontweight='bold', pad=20)
axes[0].set_xlabel('Diagnosis', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_xticklabels(['Benign (B)', 'Malignant (M)'], fontsize=12)
axes[0].set_ylim(0, max(target_counts.values) * 1.2)

# --- Pie Chart ---
wedges, texts, autotexts = axes[1].pie(
    target_counts.values,
    labels=['Benign (B)', 'Malignant (M)'],
    colors=color_list,
    autopct='%1.1f%%',
    startangle=90,
    explode=(0, 0.05),  # Slightly separate Malignant slice
    shadow=True,
    textprops={'fontsize': 12}
)
autotexts[0].set_fontweight('bold')
autotexts[1].set_fontweight('bold')
axes[1].set_title('Diagnosis Distribution (%)', fontsize=14, fontweight='bold')

plt.suptitle('🎯 Target Variable Analysis', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()

# Save to images/
plt.savefig('../images/target_distribution.png', dpi=150, bbox_inches='tight')
print("✅ Chart saved to: images/target_distribution.png")

plt.show()

### 💡 Observation & Business Insight

**Observation:**
- Benign cases (~63%) outnumber Malignant cases (~37%)
- The ratio is approximately 1.7:1 — this is a **moderate imbalance**

**Business Insight:**
- This matches real-world data — most breast masses ARE benign
- The imbalance is not severe enough to require SMOTE or undersampling
- However, we MUST use **stratified splitting** to preserve this ratio in train/test sets
- We should also use **class weights** in our models to penalize missing Malignant cases

**🎯 Interview Question:** *What is SMOTE and when would you use it?*
> SMOTE (Synthetic Minority Oversampling Technique) creates synthetic examples of the minority class by interpolating between existing minority samples. Use it when class imbalance is severe (e.g., 95:5 ratio) and you have limited data. However, it should only be applied to the training set, never the test set.

---

## 9. 📊 Basic Statistics Deep Dive

Now let's compare feature statistics **between Malignant and Benign groups**.
This gives us our first clue about which features are most discriminative.

In [ ]:
# =============================================================================
# CELL 11: Grouped Statistics — Benign vs Malignant
# =============================================================================
# WHY: Comparing means between groups reveals which features differ
#       most between Malignant and Benign. Features with large differences
#       are likely to be strong predictors.
#
# INSIGHT: If 'radius_mean' is much larger for Malignant tumors,
#       it means malignant tumors tend to be bigger — which makes
#       medical sense (cancer cells grow uncontrollably).
# =============================================================================

print_section_header("Mean Values by Diagnosis", "📊")

# Calculate mean for each group
feature_cols = [col for col in df.columns if col not in ['id', 'diagnosis']]
grouped_means = df.groupby('diagnosis')[feature_cols].mean().T

# Add a column showing the ratio (M/B)
grouped_means['M/B Ratio'] = grouped_means['M'] / grouped_means['B']
grouped_means['Diff (%)'] = ((grouped_means['M'] - grouped_means['B']) / grouped_means['B'] * 100)

# Sort by the absolute difference to find most discriminative features
grouped_means = grouped_means.sort_values('M/B Ratio', ascending=False)

print("Top 10 features with HIGHEST Malignant-to-Benign ratio:")
print("(These features are much larger in malignant tumors)")
print(grouped_means.head(10).to_string())

print("\n\nTop 5 features with LOWEST Malignant-to-Benign ratio:")
print("(These features are similar between groups — less useful for prediction)")
print(grouped_means.tail(5).to_string())

In [ ]:
# =============================================================================
# CELL 12: Interactive Feature Comparison Chart
# =============================================================================
# WHY: An interactive plotly chart lets you hover over each feature
#       to see exact values. This is much more useful than a static table.
#
# VISUALIZATION CHOICE: Horizontal bar chart — good for comparing
#       many categories (30 features) with two groups each.
# =============================================================================

# Select top 15 most discriminative features (by M/B ratio)
top_features = grouped_means.head(15).index.tolist()

# Create grouped bar chart with plotly
fig = go.Figure()

# Get means for top features
benign_means = df[df['diagnosis'] == 'B'][top_features].mean()
malignant_means = df[df['diagnosis'] == 'M'][top_features].mean()

# Normalize to percentage of max for comparability
max_vals = df[top_features].max()
benign_pct = (benign_means / max_vals * 100)
malignant_pct = (malignant_means / max_vals * 100)

fig.add_trace(go.Bar(
    y=top_features,
    x=benign_pct,
    name='Benign',
    orientation='h',
    marker_color='#3fb950',
    text=[f'{v:.1f}%' for v in benign_pct],
    textposition='auto'
))

fig.add_trace(go.Bar(
    y=top_features,
    x=malignant_pct,
    name='Malignant',
    orientation='h',
    marker_color='#f85149',
    text=[f'{v:.1f}%' for v in malignant_pct],
    textposition='auto'
))

fig.update_layout(
    title='🔬 Top 15 Most Discriminative Features (Normalized Mean %)',
    xaxis_title='% of Maximum Value',
    yaxis_title='Feature',
    barmode='group',
    height=600,
    template='plotly_dark',
    legend=dict(x=0.7, y=0.1)
)

fig.show()

# Save static version
try:
    fig.write_image('../images/feature_comparison_benign_vs_malignant.png', scale=2)
    print("✅ Chart saved to: images/feature_comparison_benign_vs_malignant.png")
except Exception as e:
    print(f"⚠️ Could not save static image: {e}")

### 💡 Key Observations

From the feature comparison above, we can see that **malignant tumors** tend to have:

- **Larger values** for: `concave points_worst`, `concavity_worst`, `perimeter_worst`, `area_worst`, `radius_worst`
- **Similar values** for: `fractal_dimension_mean`, `symmetry_mean`, `smoothness_mean`

**Medical interpretation:**
- Cancer cells have more irregular shapes (higher concavity and concave points)
- Cancer tumors are physically larger (higher radius, area, perimeter)
- The "worst" measurements (mean of 3 largest nuclei) are the most discriminative

**Conclusion:** Features describing tumor **size** and **shape irregularity** are the
strongest predictors of malignancy. This aligns with medical knowledge.

---

## 10. 📦 Outlier Detection (Initial)

### What are outliers?

Outliers are data points that are **significantly different** from other observations.
They can be:
- **Genuine extreme values** — a very large tumor (keep it!)
- **Data entry errors** — typo: 999.9 instead of 9.99 (fix or remove)
- **Measurement errors** — faulty equipment (investigate)

### Detection methods:

| Method | How it works | When to use |
|--------|-------------|------------|
| **IQR Method** | Q1 - 1.5×IQR to Q3 + 1.5×IQR | General purpose |
| **Z-Score** | Points > 3 std from mean | Normally distributed data |
| **Visual (Boxplot)** | Points beyond whiskers | Quick exploration |

In [ ]:
# =============================================================================
# CELL 13: Outlier Detection using IQR Method
# =============================================================================
# WHY: Identifying outliers early helps us decide whether to:
#       - Keep them (if they represent real extreme cases)
#       - Cap them (winsorization)
#       - Remove them (if they're data errors)
#       - Use robust methods (median instead of mean)
#
# IQR METHOD: A point is an outlier if it's below Q1 - 1.5*IQR
#       or above Q3 + 1.5*IQR. This is what boxplots use.
# =============================================================================

print_section_header("Outlier Detection (IQR Method)", "📦")

def count_outliers_iqr(df: pd.DataFrame, columns: list) -> pd.DataFrame:
    """
    Count outliers in each column using the IQR method.
    
    An outlier is defined as a value below Q1 - 1.5*IQR
    or above Q3 + 1.5*IQR.
    
    Args:
        df: Input DataFrame
        columns: List of numeric column names
    
    Returns:
        DataFrame with outlier counts and percentages
    """
    outlier_info = []
    
    for col in columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        
        outliers = df[(df[col] < lower_bound) | (df[col] > upper_bound)]
        
        outlier_info.append({
            'Feature': col,
            'Outlier Count': len(outliers),
            'Outlier %': len(outliers) / len(df) * 100,
            'Lower Bound': round(lower_bound, 4),
            'Upper Bound': round(upper_bound, 4),
            'Min': round(df[col].min(), 4),
            'Max': round(df[col].max(), 4)
        })
    
    return pd.DataFrame(outlier_info).sort_values('Outlier Count', ascending=False)


# Run outlier detection
outlier_summary = count_outliers_iqr(df, feature_cols)

# Show features with outliers
with_outliers = outlier_summary[outlier_summary['Outlier Count'] > 0]
print(f"Features with outliers: {len(with_outliers)} out of {len(feature_cols)}")
print(f"\nTop 10 features by outlier count:")
print(with_outliers.head(10).to_string(index=False))

# Total outlier impact
total_outliers = with_outliers['Outlier Count'].sum()
total_points = len(df) * len(feature_cols)
print(f"\n📊 Total outlier data points: {total_outliers} out of {total_points:,} ({total_outliers/total_points*100:.2f}%)")

In [ ]:
# =============================================================================
# CELL 14: Boxplot Visualization for Top Features
# =============================================================================
# WHY: Boxplots visually show the distribution and outliers.
#       The whiskers extend to 1.5*IQR, and points beyond them
#       are marked as individual dots (outliers).
#
# We show only the _mean features (10 features) for clarity.
# =============================================================================

# Select only _mean features for cleaner visualization
mean_features = [col for col in feature_cols if 'mean' in col]

fig, axes = plt.subplots(2, 5, figsize=(20, 10))
axes = axes.flatten()

for i, feature in enumerate(mean_features):
    # Create boxplot colored by diagnosis
    data_B = df[df['diagnosis'] == 'B'][feature]
    data_M = df[df['diagnosis'] == 'M'][feature]
    
    bp = axes[i].boxplot(
        [data_B, data_M],
        labels=['Benign', 'Malignant'],
        patch_artist=True,
        widths=0.6
    )
    
    # Color the boxes
    bp['boxes'][0].set_facecolor('#3fb950')
    bp['boxes'][1].set_facecolor('#f85149')
    bp['boxes'][0].set_alpha(0.7)
    bp['boxes'][1].set_alpha(0.7)
    
    # Clean feature name for title
    clean_name = feature.replace(' mean', '').replace('_', ' ').title()
    axes[i].set_title(clean_name, fontsize=11, fontweight='bold')
    axes[i].tick_params(axis='x', labelsize=9)

plt.suptitle('📦 Boxplots of Mean Features by Diagnosis\n(Points beyond whiskers are outliers)',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()

# Save
plt.savefig('../images/boxplots_mean_features.png', dpi=150, bbox_inches='tight')
print("✅ Chart saved to: images/boxplots_mean_features.png")

plt.show()

### 💡 Outlier Analysis & Decision

**Observations:**
- Most features have some outliers (typical for medical data)
- Malignant tumors tend to have MORE extreme values (their distributions extend higher)
- The outliers appear to be **genuine extreme measurements**, not data errors

**Decision:** We will **NOT remove** these outliers because:
1. They represent real extreme cancer cases — removing them would remove important signal
2. The outliers are in the direction that makes medical sense (large tumors = more likely cancer)
3. We'll use **StandardScaler** which is somewhat sensitive to outliers, but the effect is manageable
4. Tree-based models (Random Forest, XGBoost) are naturally robust to outliers

**🎯 Interview Question:** *How do you decide whether to remove or keep outliers?*
> It depends on the domain context. If outliers are data errors (e.g., age=999), remove them. If they represent genuine extreme cases (e.g., very large tumors), keep them — they contain valuable signal. Use boxplots and domain knowledge to make this determination. In medical data, extreme values are often the most important cases.

---

## 11. 📋 Phase 2 Summary

### ✔ Summary

In Phase 2, we thoroughly explored the Wisconsin Breast Cancer dataset:
- Loaded 569 samples with 30 features + 1 target + 1 ID
- All 30 features are numeric (float64) — no encoding needed for features
- **No missing values** — dataset is 100% complete
- **No duplicate rows** — every sample is unique
- **Moderate class imbalance** (~63% Benign, ~37% Malignant) — use stratified split
- **Outliers present** but are genuine extreme values — keep them
- Features related to tumor **size** and **shape irregularity** are most discriminative

### ✔ What You Learned

| Concept | Description |
|---------|-------------|
| PEP-8 Import Order | Standard → Third-party → Local |
| `.info()` vs `.describe()` | Structure vs. statistics |
| Class Imbalance | When one class dominates the dataset |
| IQR Outlier Method | Q1 - 1.5×IQR to Q3 + 1.5×IQR |
| MCAR/MAR/MNAR | Types of missing data patterns |
| Feature Types | Numeric, categorical, ordinal, binary |

### ✔ Files Created

```
📁 Cancer Prediction using Machine Learning/
├── 📁 data/raw/
│   └── 📄 breast_cancer.csv              ← Raw dataset (golden copy)
├── 📁 images/
│   ├── 🖼️ target_distribution.png         ← Class distribution chart
│   ├── 🖼️ feature_comparison_benign_vs_malignant.png
│   └── 🖼️ boxplots_mean_features.png      ← Outlier visualization
└── 📁 notebooks/
    └── 📓 01_data_exploration.ipynb        ← Phase 1 + 2 (this notebook)
```

### ✔ Next Phase Preview: Phase 3 — Data Cleaning

In the next phase, we will:
1. Drop the `id` column (not useful for prediction)
2. Encode the target: M → 1, B → 0
3. Verify no further cleaning is needed
4. Save cleaned data to `data/processed/breast_cancer_cleaned.csv`
5. Create the reusable `src/preprocessing.py` module

---

*Phase 2 Complete ✅ — Proceeding to Phase 3*